### **¿Qué es Unity Catalog?**

Imagina Unity Catalog como una capa de control maestra que se sitúa por encima de todos tus workspaces de Databricks y tu almacenamiento en la nube (S3, ADLS, GCS).

- **Antes (Hive Metastore):** Los permisos y tablas vivían dentro de un Workspace. Si tenías 3 workspaces (Dev, QA, Prod), tenías que crear los usuarios y permisos 3 veces.

- **Ahora (Unity Catalog):** Defines los datos y los permisos una sola vez a nivel de cuenta. Múltiples workspaces pueden acceder a los mismos datos según los permisos que tú definas centralmente.

### **El Modelo de Objetos (La Jerarquía de 3 Niveles)**

Para entender UC, debes entender cómo organiza los datos. Utiliza un espacio de nombres de tres niveles (`catalog`.`schema`.`table`) que reemplaza al antiguo `database`.`table`.

- **Metastore:** El contenedor de nivel superior (generalmente uno por región/nube).

- **Catálogo (Catalog):** El primer nivel de agrupación de datos. Ejemplo: `prod_catalog`, `rrhh_catalog`. Es la frontera más dura para el aislamiento de datos.

- **Esquema (Schema):** (Antes llamado Base de Datos). Contiene las tablas, vistas, volúmenes y modelos. Ejemplo: `finance_schema`.

![](https://docs.databricks.com/aws/en/assets/images/object-model-40d730065eefed283b936a8664f1b247.png)

**Activos (Assets):**

- **Tablas:** Datos estructurados (Delta Tables).

- **Vistas:** Consultas guardadas.

- **Volúmenes:** Datos no estructurados (PDFs, imágenes, CSVs, JSONs brutos).

- **Modelos:** Modelos de IA/ML registrados en MLflow.

- **Funciones:** User Defined Functions (UDFs).

### **Pilares del Gobierno de Datos en Unity Catalog**

El verdadero valor de UC reside en sus capacidades de gobierno. Aquí están los detalles técnicos:

**A. Gestión de Identidad y Acceso (IAM Unificado)**

UC separa la identidad del workspace.

- **Sincronización:** Se integra con tu proveedor de identidad (Azure Entra ID, Okta, AWS IAM).

- **Service Principals:** Soporte nativo para cuentas de servicio (automatización/jobs) que no dependen de un usuario humano.

**B. Control de Acceso (ACLs) Estándar SQL**

Olvídate de gestionar permisos con archivos JSON complejos o políticas de IAM de la nube difíciles de mantener. UC usa SQL estándar ANSI para todo:

### **Privilegios**

In [0]:
%sql
SHOW GRANTS `<email>` ON CATALOG workspace;


## **Tipos de Privilegio por Objeto**

#### **Metastore**

`CREATE CATALOG`, `CREATE CLEAN ROOM`, `CREATE CONNECTION`, `CREATE EXTERNAL LOCATION`, `CREATE EXTERNAL METADATA`, `CREATE PROVIDER`, `CREATE RECIPIENT`, `CREATE SHARE`,`CREATE SERVICE CREDENTIAL`, `CREATE STORAGE CREDENTIAL`, `SET SHARE PERMISSION`, `USE MARKETPLACE ASSETS`, `USE PROVIDER`, `USE RECIPIENT`, `USE SHARE`

#### **Catalog**

`ALL PRIVILEGES`, `APPLY TAG`, `BROWSE`, `CREATE SCHEMA`, `USE CATALOG`

All users have `USE CATALOG` on the main catalog by default.

The following privilege types apply to securable objects in a catalog. You can grant these privileges at the catalog level to apply them to current and future objects in the catalog.

`CREATE FUNCTION`, `CREATE TABLE`, `CREATE MATERIALIZED VIEW`, `CREATE MODEL`, `CREATE VOLUME`, `EXTERNAL USE SCHEMA`, `READ VOLUME`, `REFRESH`, `WRITE VOLUME`, `EXECUTE`, `MANAGE`, `MODIFY`, `SELECT`, `USE SCHEMA`

#### **Schema**

`ALL PRIVILEGES`, `APPLY TAG`, `CREATE FUNCTION`, `CREATE TABLE`, `CREATE MODEL`, `CREATE VOLUME`, `CREATE MATERIALIZED VIEW`, `MANAGE`, `EXTERNAL USE SCHEMA`, `USE SCHEMA`

The following privilege types apply to securable objects within a schema. You can grant these privileges at the schema level to apply them to current and future objects within the schema.

`EXECUTE`, `MODIFY`, `READ VOLUME`, `REFRESH`, `SELECT`, `WRITE VOLUME`

#### **Table**

`ALL PRIVILEGES`, `APPLY TAG`, `MANAGE`, `MODIFY`, `SELECT`

#### **Materialized View**

`ALL PRIVILEGES`, `APPLY TAG`, `MANAGE`, `REFRESH`, `SELECT`

#### **View**

`ALL PRIVILEGES`, `APPLY TAG`, `MANAGE`, `SELECT`

#### **Volume**

`ALL PRIVILEGES`, `MANAGE`, `READ VOLUME`, `WRITE VOLUME`

#### **External Location**

`ALL PRIVILEGES`, `BROWSE`, `CREATE EXTERNAL TABLE`, `CREATE EXTERNAL VOLUME`, `CREATE FOREIGN SECURABLE`, `CREATE MANAGED STORAGE`, `EXTERNAL USE LOCATION`, `MANAGE`, `READ FILES`, `WRITE FILES`

#### **External Metadata**

`ALL PRIVILEGES`, `BROWSE`, `MANAGE`, `MODIFY`

#### **Service Credential**

`ALL PRIVILEGES`, `ACCESS`, `CREATE CONNECTION`, `MANAGE`

#### **Storage Credential**

`ALL PRIVILEGES`, `CREATE EXTERNAL LOCATION`, `CREATE EXTERNAL TABLE`, `MANAGE`, `READ FILES`, `WRITE FILES`

#### **Connection**

`ALL PRIVILEGES`, `CREATE FOREIGN CATALOG`, `MANAGE`, `USE CONNECTION`

#### **Function**

`ALL PRIVILEGES,` `APPLY TAG` (models only), `CREATE MODEL VERSION` (models only), `EXECUTE`, `MANAGE`

## **Tipos de Privilegio**

### **`ALL PRIVILEGES`**

**Tipos de objetos aplicables:**

`CATALOG`, `EXTERNAL LOCATION`, `EXTERNAL METADATA`, `SERVICE CREDENTIAL`, `STORAGE CREDENTIAL`, `SCHEMA`, `FUNCTION` (incluyendo modelos), `PROCEDURE`, `TABLE`, `MATERIALIZED VIEW`, `VIEW`, `VOLUME`.

Se utiliza para conceder o revocar _**todos los privilegios aplicables al objeto securizable y a sus objetos hijos**_, sin necesidad de especificarlos de manera explícita.

Cuando se concede `ALL PRIVILEGES` sobre un objeto, no se otorgan individualmente al usuario cada uno de los privilegios aplicables en ese momento. En cambio, se expande a todos los privilegios disponibles en el momento en que se realizan las verificaciones de permisos. Esto significa que, a medida que Databricks introduce nuevos privilegios y nuevos objetos securizables, un `ALL PRIVILEGES` existente incluye automáticamente cualquier privilegio nuevo aplicable al objeto, a sus objetos hijos existentes y a cualquier nuevo objeto hijo.

Para evitar una exfiltración accidental de datos o una escalada de privilegios, `ALL PRIVILEGES` no incluye los privilegios `EXTERNAL USE SCHEMA`, `EXTERNAL USE LOCATION` ni `MANAGE`.

Cuando se **revoca** `ALL PRIVILEGES`, tanto la concesión de `ALL PRIVILEGES` como todos los privilegios individuales que implicaba se eliminan. Los privilegios que no forman parte de `ALL PRIVILEGES` —como `MANAGE`, `EXTERNAL USE LOCATION` y `EXTERNAL USE SCHEMA`— no se ven afectados.

### **CREATE CATALOG**

**Tipos de objetos aplicables:** Unity Catalog metastore

Permite que un usuario cree un catálogo dentro de un metastore de Unity Catalog.

Para crear un catálogo externo (foreign catalog), también se debe tener el privilegio `CREATE FOREIGN CATALOG` sobre la conexión que contiene ese catálogo externo o sobre el propio metastore.

### **CREATE SCHEMA**

**Tipos de objetos aplicables:** `CATALOG`

Permite que un usuario cree un esquema (schema).

- El usuario también debe tener el privilegio `USE CATALOG` sobre el catálogo en el que desea crear el esquema.

### **CREATE VOLUME**

- **Tipos de objetos aplicables:** `SCHEMA`

Permite que un usuario cree un volumen dentro de un esquema.

Dado que los privilegios se heredan, `CREATE VOLUME` también puede otorgarse sobre un catálogo, lo que permite al usuario crear volúmenes en cualquier esquema existente o futuro dentro del catálogo.

El usuario también debe tener el privilegio `USE CATALOG` sobre el catálogo padre y `USE SCHEMA` sobre el esquema padre.

### **CREATE TABLE**

**Tipos de objetos aplicables:** `SCHEMA`

Permite que un usuario cree una tabla o vista dentro de un esquema.

Dado que los privilegios se heredan, `CREATE TABLE` también puede otorgarse sobre un catálogo, lo que permite al usuario crear tablas o vistas en cualquier esquema existente o futuro dentro del catálogo.

- El usuario también debe tener el privilegio `USE CATALOG` sobre el catálogo padre y `USE SCHEMA` sobre el esquema padre.

### **CREATE CONNECTION**

**Applicable object types:** Unity Catalog metastore, `SERVICE CREDENTIAL`

Permite que un usuario cree una conexión a una base de datos externa en un escenario de Lakehouse Federation.

Para usar una credencial de servicio al momento de crear la conexión, el usuario debe tener este privilegio tanto en el metastore como en la propia credencial de servicio.

### **ACCESS**

**Tipos de objetos aplicables:** `SERVICE CREDENTIAL`

Permite que un usuario utilice una credencial de servicio para acceder a un servicio externo o a varios servicios externos.

### **APPLY TAG**

**Tipos de objetos aplicables:** `CATALOG`, `SCHEMA`, `TABLE`, `VOLUME`, `MATERIALIZED VIEW`, `VIEW`, y modelos registrados como `FUNCTION`.

Permite que un usuario agregue y edite tags en un objeto. Conceder `APPLY TAG` sobre una tabla o vista también habilita el tagging de columnas. Conceder `APPLY TAG` a un modelo registrado también habilita el tagging de versiones del modelo.

El usuario también debe tener el privilegio `USE CATALOG` en el catálogo padre y `USE SCHEMA` en el esquema padre.

Para aplicar un governed tag a objetos securizables de Unity Catalog, también es necesario contar con el permiso `ASSIGN` sobre dicho governed tag.

### **CREATE EXTERNAL LOCATION**

- **Tipos de objetos aplicables:** Unity Catalog METASTORE, `STORAGE CREDENTIAL`

Para crear una ubicación externa (external location), el usuario debe tener este privilegio tanto en el metastore como en la credencial de almacenamiento (storage credential) que se referencia en dicha ubicación externa.

## **`GRANT`**



In [0]:
%sql
SHOW GRANTS [ principal ] ON securable_object;

In [0]:
%sql
GRANT privilege(s) ON securable_object TO principal;

In [0]:
%sql
-- Dar permiso de lectura a un grupo
GRANT SELECT ON TABLE catalogo.esquema.tabla TO `grupo_analistas`;

-- Permitir crear tablas en un esquema
GRANT CREATE TABLE ON SCHEMA catalogo.esquema TO `data_engineers`;

## **`REVOKE`**

In [0]:
%sql
REVOKE privilege_types ON securable_object FROM principal;

In [0]:
%sql
REVOKE SELECT ON TABLE t FROM aliens;

## **GROPUS**

### **CREATE GROUP**

Crea un grupo local del área de trabajo con el nombre especificado, **opcionalmente incluyendo una lista de usuarios y grupos**. Los grupos locales del área de trabajo no se sincronizan con la cuenta de Databricks y no son compatibles con Unity Catalog. Para obtener más información, consulta Administrar grupos locales del área de trabajo (heredado).

- Para ejecutar este comando debes ser **administrador**.

In [0]:
%sql
-- Create an empty group.
CREATE GROUP humans;

-- Create tv_aliens with Alf and Thor as members.
CREATE GROUP tv_aliens WITH USER `alf@melmak.et`, `thor@asgaard.et`;

-- Create aliens with Hilo and tv_aliens as members.
CREATE GROUP aliens WITH USER `hilo@jannus.et` GROUP tv_aliens;

## **DROP GROUP**

Elimina un grupo local del área de trabajo. Se lanzará una excepción si el grupo no existe en el sistema.

Los grupos locales del área de trabajo no se sincronizan con la cuenta de Databricks y no son compatibles con Unity Catalog. Para obtener más información, consulta Administrar grupos locales del área de trabajo (heredado).

- Para ejecutar este comando debes ser **administrador**.

In [0]:
%sql
DROP GROUP group_principal;

# **Gestión del Almacenamiento: External vs Managed**

Unity Catalog cambia cómo Databricks interactúa con tu almacenamiento en la nube (S3/ADLS).

**Credenciales de Almacenamiento y Ubicaciones Externas**

Ya no se deben hardcodear claves de acceso (Access Keys) en los notebooks.

- **Storage Credential:** Un objeto seguro en UC que encapsula un rol de IAM (AWS) o Service Principal (Azure).

- **External Location:** Una ruta de almacenamiento (ej. `s3://my-bucket/finance/`) autorizada para usar una Credencial específica.

**Tipos de Tablas**

- **Managed Tables (Tablas Gestionadas):** Databricks gestiona tanto los metadatos como los archivos físicos en el almacenamiento raíz del metastore. Si borras la tabla, se borran los datos. Recomendado para la mayoría de los casos.

- **External Tables (Tablas Externas):** Databricks gestiona los metadatos, pero tú controlas la ubicación física de los archivos. Si borras la tabla en UC, los archivos en S3/ADLS permanecen. Útil si otras herramientas fuera de Databricks necesitan leer esos archivos directamente.

![](https://cdn.prod.website-files.com/66954b344e907bd91f1c8027/6835e83fff2e3b864fa1941a_AD_4nXfDKyo3WXqPNPYDnNkIb_Br1SR8ueQoQWA5JvZzP5GHlB_zeNGEdUQNjpdJqBb447OzLQgeygdcKR4ygZVA_1-heWgDOq447fioCXnMTK62rXLM4szHwDGyJvvPxYWjFiXp_bStzw.png)

| Característica | **Tabla Managed** (Gestionada) | **Tabla External** (Externa) |
| :--- | :--- | :--- |
| **Control** | Databricks controla tanto los datos como los metadatos | Databricks controla solo los metadatos |
| **Ubicación de datos** | Ubicación predeterminada gestionada por Databricks | Ubicación de almacenamiento especificada por el usuario (`LOCATION`) |
| **Eliminación** | `DROP TABLE` elimina **datos y metadatos** | `DROP TABLE` elimina solo los **metadatos**; los datos persisten |
| **Creación** | Opción predeterminada (`CREATE TABLE ...`) | Requiere la palabra clave `EXTERNAL` y `LOCATION` |

In [0]:
%sql
CREATE EXTERNAL TABLE 
  mi_tabla_external 
(
id INT, 
nombre STRING
) 
LOCATION 
  's3://mi-bucket/datos/mi_tabla_external'